In [0]:
%run "/Users/scottsbv@gmail.com/student-engagement/Dea Student Engagement Pipeline/silver_engine_utils"

In [0]:
# ============================================================================
# SILVER CHILD TABLE — lead_activites_raw_user_note_mentions
# ============================================================================
# Purpose: User note mention references from activities
# This table is NEEDED when:
#   - You need to find all user mentions in activity notes
#   - You're analyzing user engagement and collaboration
#   - You're tracking who was mentioned in activity notes
# ============================================================================

from pyspark.sql.functions import col, explode_outer

parent_table = "crm_ingestion.silver.lead_activites_raw"
child_table  = "crm_ingestion.silver.lead_activites_raw_user_note_mentions"

print("\n" + "="*80)
print("🔥 BUILDING CHILD TABLE — lead_activites_raw_user_note_mentions")
print("="*80)

df_parent = spark.table(parent_table)

# Explode the user_note_mentions array into individual rows
df_child = (
    df_parent
        .filter(col("user_note_mentions").isNotNull())
        .withColumn("user_note_mentions", explode_outer(col("user_note_mentions")))
        .select(
            col("id").alias("lead_activites_raw_id"),
            col("user_note_mentions"),
            col("bronze_insert_date"),
            col("bronze_updated_at")
        )
)

df_child.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(child_table)

print(f"\n✅ Table created: {child_table}")
print(f"🔢 Row count: {df_child.count():,}")

print("\n📘 SCHEMA:")
df_child.printSchema()

print("\n📊 SAMPLE ROWS (5):")
display(spark.table(child_table).limit(5))